# 05F · Locked literature and experimental synthesis

**Purpose:** close Experiment 05 without changing any Stage-05E threshold, hypothesis,
endpoint, or test. This notebook validates the completed 05E archive, combines it with
the literature snapshot locked to **15 September 2026**, and writes the final claim
boundary and manuscript-ready reporting package.

This is a reporting and decision notebook, not a new experiment. It performs no model
fitting, no threshold tuning, no alternative hypothesis testing, and no post-test rescue.

**Expected valid outcome:** the notebook may conclude that the broad novelty claim is not
established while retaining a narrower, independently supported acquisition-chain
contribution. That is a completed scientific result, not a runtime failure.

**Runtime:** CPU only; approximately one minute after Drive mounts.



## 1. Setup and immutable inputs

Leave the archive field blank to search the project `results/` folder. If the original
notebook-generated 05E ZIP is present, it is selected by its recorded SHA-256. A Google
Drive folder re-ZIP is also accepted when every internal file matches the 05E export
manifest.



In [ ]:
import base64
import csv
import gzip
import hashlib
import io
import json
import os
import shutil
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import Image as DisplayImage, display
except ImportError:
    DisplayImage = None
    def display(value):
        print(value)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/reliable-reconstruction-under-mismatch')
else:
    PROJECT_DIR = Path(os.environ.get('RRM_PROJECT_DIR', Path.cwd())).resolve()

RESULTS_DIR = PROJECT_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

ANALYSIS_ARCHIVE_TEXT = os.environ.get('RRM_05E_ARCHIVE', '')  #@param {type:"string"}
EXPECTED_ORIGINAL_05E_SHA256 = '6566319869d7b2c86902aa4f29d071c8a7bdbbce2bbe1b5a9649f4aaab690fa3'
EXPECTED_GAP_MAP_SHA256 = '873701f82aed5cb244d222dbb9544fa3d6da5801c9e87ce350479106a9d6a5fb'
EXPECTED_REVIEW_MATRIX_SHA256 = '3291b46520bcca12f0c1caf971fe9b0852ecf8e7662df453e7ed2b4a5bc4c5fd'
LITERATURE_CUTOFF = '2026-09-15'
PRIMARY_CHAIN = 'j75_b16_n2'

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def show_table(frame, rows=20):
    frame = frame.head(rows)
    try:
        display(frame)
    except Exception:
        print(frame.to_string(index=False))

print(json.dumps({
    'in_colab': IN_COLAB,
    'project_dir': str(PROJECT_DIR),
    'literature_cutoff': LITERATURE_CUTOFF,
    'gpu_required': False,
}, indent=2))



## 2. Locate and fully validate the Stage-05E archive

The validation is content-based. It supports both the original flat archive and a Drive
wrapper ZIP containing an `independent_05e_locked_analysis/` top-level directory.



In [ ]:
def inspect_05e_archive(path):
    path = Path(path)
    result = {
        'path': str(path),
        'archive_sha256': sha256_file(path),
        'archive_byte_count': path.stat().st_size,
        'valid': False,
    }
    with zipfile.ZipFile(path) as archive:
        bad_member = archive.testzip()
        assert bad_member is None, f'CRC failure in {path.name}: {bad_member}'
        names = set(archive.namelist())
        manifest_members = [name for name in names if name.endswith('export_manifest.json')]
        candidates = []
        for member in manifest_members:
            try:
                manifest = json.loads(archive.read(member))
            except Exception:
                continue
            if manifest.get('stage') == '05E_one_time_locked_analysis':
                prefix = member[:-len('export_manifest.json')]
                candidates.append((prefix, member, manifest))
        assert len(candidates) == 1, f'Expected one Stage-05E manifest in {path.name}, found {len(candidates)}'
        prefix, manifest_member, manifest = candidates[0]
        checked = 0
        for row in manifest['files']:
            member = prefix + row['path']
            assert member in names, f'Missing manifested member: {member}'
            payload = archive.read(member)
            assert len(payload) == int(row['byte_count']), f'Byte-count mismatch: {member}'
            assert sha256_bytes(payload) == row['sha256'], f'SHA-256 mismatch: {member}'
            checked += 1
        status = json.loads(archive.read(prefix + 'status.json'))
        primary = json.loads(archive.read(prefix + 'primary_hypotheses.json'))
        checks = json.loads(archive.read(prefix + 'checks.json'))
        receipt = json.loads(archive.read(prefix + 'unsealing_receipt.json'))
        assert status['status'] == 'passed_one_time_locked_analysis'
        assert status['independent_test_run_complete'] is True
        assert status['independent_test_performance_inspected'] is True
        assert status['unsealed'] is True
        assert primary['experimental_novelty_gate_passed'] is False
        assert checks['sources'] == 40 and checks['sealed_shards'] == 5
        assert receipt['sealed_shards_verified_before_unsealing'] is True
        assert receipt['test_performance_inspected_before_unsealing'] is False
        result.update({
            'valid': True,
            'prefix': prefix,
            'manifest_member': manifest_member,
            'manifest_files_checked': checked,
            'internal_manifest_sha256': sha256_bytes(archive.read(manifest_member)),
        })
    return result

if ANALYSIS_ARCHIVE_TEXT.strip():
    archive_candidates = [Path(ANALYSIS_ARCHIVE_TEXT).expanduser()]
else:
    archive_candidates = sorted(RESULTS_DIR.rglob('independent_05e_locked_analysis*.zip'))
    archive_candidates += sorted((PROJECT_DIR / 'upload').glob('independent_05e_locked_analysis*.zip'))
    archive_candidates = list(dict.fromkeys(path.resolve() for path in archive_candidates if path.is_file()))

assert archive_candidates, (
    'No Stage-05E ZIP found. Put the original independent_05e_locked_analysis_<timestamp>.zip '
    'inside the project results/ folder, or paste its complete path above.'
)

inspected_archives = []
inspection_errors = []
for candidate in archive_candidates:
    try:
        inspected_archives.append(inspect_05e_archive(candidate))
    except Exception as error:
        inspection_errors.append({'path': str(candidate), 'error': repr(error)})

valid_archives = [row for row in inspected_archives if row['valid']]
assert valid_archives, f'No valid Stage-05E archive. Errors: {inspection_errors}'
originals = [row for row in valid_archives if row['archive_sha256'] == EXPECTED_ORIGINAL_05E_SHA256]
if originals:
    selected_archive = originals[0]
else:
    internal_hashes = {row['internal_manifest_sha256'] for row in valid_archives}
    assert len(internal_hashes) == 1, (
        'Multiple non-equivalent valid 05E archives were found. Paste the intended full ZIP path above.'
    )
    selected_archive = valid_archives[0]

ANALYSIS_ARCHIVE_05E = Path(selected_archive['path'])
print(json.dumps({
    'selected_archive': str(ANALYSIS_ARCHIVE_05E),
    'outer_sha256': selected_archive['archive_sha256'],
    'is_original_notebook_zip': selected_archive['archive_sha256'] == EXPECTED_ORIGINAL_05E_SHA256,
    'manifest_files_checked': selected_archive['manifest_files_checked'],
    'content_equivalent_valid_archives_found': len(valid_archives),
}, indent=2))



## 3. Load the locked results and literature snapshot

The 90-study evidence snapshot is embedded in compressed canonical CSV form so Colab
does not depend on Excel readers, network access, or a mutable external literature file.
Its canonical SHA-256 is checked before use. The source workbook and gap-map hashes are
recorded in the final receipt.



In [ ]:
LITERATURE_SNAPSHOT_GZIP_B64 = '''H4sIAAAAAAACA719W3Pb2K7m+/wKlh/2y5CWqAslJXXqlG9J3G07iuUku/eLiiKXLCYUqcOLE/Vvm7f5YwNgXXmTZSc9VTu7E91ILmABH4APWIuiDPfW9aX9F/Mz+6wsNmmW2w9RETP7C0tKZs/LVRwFfhGliVXsd/ACY5mVsaeI/WChfZuGfhwVe/tdmv3ws9DapiGL7dso3/pFsOFfWaTxE3xp7W+jeG/PN/s8CnIrSgr2mNEv23+k8C8r2vqPrJfuGLyaZvY1/tP6/Mn+KF7Bv1/A5VbiWx8/XloFywv7nvmxFfqF39vAPcB9MPsi3e7SMgmtEC8S8i88RFtmPfnZPkoe7Q9+HJdBlIhnw985W+UFS+ijF/Aclv/kR7G/gsW4eopClgTMygu/KHP7OgniMscvhiyIcvpGnObwI1YAV2ZFhE/w3t9ZMVsXFjxTYs8zeMBsb+VpmcEvfb6/gRvHdbSSFC7/v+b9vmsP+oO+fZkmjxYrLD8+tS8Z21k3IJ4Ebtpawyp88JMwxn/8ybKExb1bXHLrM9xdVviwjntYWosv3iUL0uQpjUv+TF/m9/BgyZpl+Cwg9Bx+fhWXWYYLcqE/CsuJL9viCiRUi2UZPBPdz8PNwiqTLI1jvpJZ2IsKFFL0BBriZ0Xkx/Zdav4VL2b8U/3vc/I9SX8ktPaZHxQWqEq0jkC3aI1Dfpt3qfWNdCQQ8mdhTV3gftQCvIUVJTmQBvjB/5QgIhJzsIEP2Jui2OVver0wjU7T7LHn9k9dtz/r4QKNBl6/f4piOO33ByMPlgXWIMuLNGHWVur1zofLosQGXGI3kZTX3dXDwxsLVR5FdJ3A4+TMmmcpqNE2t35ExcaiNbxjZQbrfMeKH2n2HVWq+lHYFGUGkqDnf0i3KejxbrPvReJz+Pi4+u8Z3F4UWGrf3OGdBqDH8PM2aQ4sVsYey9jPopzvhEW6LnqwEyKfi7siEimmQ+J6V8axFUQF3z0NmdGHYUGE1OS69T5/6q1hT5UZ7KQ9bLZtqyz602nPHXqu440G/Z6/8sLxxH7YsDSDjQUaYO24DXGiBDbEFh5v5ecMdJGhSIZcJGdJUSZSKh9hRySwd1cRWKuI5Va6tqo7S+2Ze9wzoItlQI8Gm80qNiCWlEwDXPwizQv6gbNre353tqgI6vb+unfxYC/87Y72qJLKHJSzzFa0Xj3+8yT/fBOtC/sy8h8T+F2QY4422a4sOv4w3wXyX+I/z0iBNjj/ZXhkEP53a1UWVpjCv8DmwD+iGPYHbB36HmyZLGMBbmIwUmiz26UzGfZ2iZ+fujP462TiuhMwZHHEF3cPNgp2Hd0RSmPEpXEfbaUs0Fo7X9MMLn0OmmBdguHO4T20bUocuOznYKU2YDK/4wvaUFln8WOawT6CPfKcSfO3sPzglGh9yKSRr6C/iev2VvIydusu0Auvl/957ccnc/IdOId1FJA9WuN38jJ7YlEc++hLdtGOlNbirk2uNToLPwhYnp+C4gVP61MwZT1QSlTAJT7xEpe0tym2cQ/WdakXdImXXYoHW8KCLuWCLmFBl+aCLvUyLfWC6p9fkok7xYvQSkm3FcAPRSBe2mpjFK5r/4eBVP5l3fhb+z5dleAEa5voq7J6SsBFyh0Zs7j/UpABNuINS/IYnp92JMrx+urqynq4uK5sNfUpaQnni3e9WL7IgQi8RKrkJ2T+HO7JpDkCa4Y+DDRfvuKQ9064lbQSbpsPOLi2/fkiQ4lfO5EoqqowhqsD7QDV2e3SDGwpgApQGF9hlZNOfwYrhp7MPR267mg8Gth3sPqIUeA/8V57Mw1ZUKgeF+r7KAafV4EhV+BI8Z6icmudZcEGlgStGFgT3LsNXydMaqcApeeqe7Qb2BN+1uHQ6EaYvpFeJw451qUdt6HP8rzcwpOWOQJCi4SrbvGtXsvMsIRoY+V9HyOj6Ww0ss2Frfi1CZfLArzav6zztNz6CV8NXHC6JPkPueRnZ+CfaqYRvROg4Sg9bq2/aIBgPaKwaGGtHTgqWCH4NMIIK46+wz1u0jQ0vFM7CKyseTfigw9wX6yuZKVJDLBOgT213LjEhDFAnzO2A12Ee4efUsb0W37q+z5f8ygJ2c/T3WbXw8Xp4X0FMeshBu+53mjYtz9/qrmvKV/0840fgiPRaKISPZCqK4gmZFBHEmIP3F53ADvb/AENG/DpeyEZYEIK1bDFh98BKJTb92DhWC9BNYaIBKEvR7yNVa9K42VY4sSAKesMfCtZR67mEm0rS+pjHAYyMdfggKm6ldugP5lMAe6988FngM9M0JhXZTITXmfjm2FSHj1y5AD7AbSKcOKlDv6Em0FDRZvmHLZVKAS1KGG5nXuWy0jp+uLiS33v8A9l+kOLfQL+GdfCiDFFhHEhJABAalOu1+hizED0/dld7+LuzkJjnIpo9hoRI6x/K/KreJfjDNYJxExqv6AY4EN7W3kVvKS1ZT4gEraFPaP8S4dvOYhKerhgKBQOSkgySyWSpb9UIlkaIlmSSAimoEiWJJIliWRZF8kSr4DAxK0AE2Ph82gLLh4f1jCabp/ryldDVQguXS3uQQhvrIcMxIVqU0GlqBr1O+AgZo5WWUsekRFpi/UVg7hNusuf1ZsP0ePGSbOQZRADtaiQVp7eRn+0pj7/qOpArBKwXVHCDjIuY8V+8D03Yck6Avmho0M1AC+cv1BbvvbOrm+5yqCAloZkllIwVXSLCtJQDRTMEgWzVIIhDEw687WhNO+1JzOfztQaysMMIIR8ZHEKdi1Oc6k9pJ5OkTrSxj/Cj1AmiIf24ISzaEU35rzLGKskZj6VPmyrtUyoITI92+1kgq2KmC5ub9rc9zYKsjQP0t2+t7i3H/z8uxMyWO0QFta+NRI1+F0IjkGGgB5rtr4Z29QB0rHRpQwlI7xRCCSruCh/a2GS7glUJiRAxFiIGpQxsOyg0mgglM7ssjSA9+Hh89NtnJ2iM897T6436PmGIAYDn8vRSAVWHYQ74OL7E12QdhBJGuUUSUbrNc/d3Rvip6WD1WBldj1fNFa+jItoB7hPYlWIsL7nzyBVdR0NY8xdS3AVET99ugGUfh2yLiCQiUjPIdiKEhHdG6A1MTyEEYCagkhgPaJdfhoEPdpCy3UE4RX/ew9Xubfx801vNh73R+PZeOW53swduiuPhV4QTgZs1R/Da44Ee45eWGHC1SqZG3DIJajTamdx7ESJ8xHWV248LTt06iLesi7gKctdZ76TBOmg9FotKKVA+NctWobckL3xHua1cwg2n1BnLtD6+TmttAkF4KKgVFm3eW5NMRyLxcC9s5/irpUU4ZEA2XPnXokf0a1LD4+57mMdO64hFzNa6ZtoqQUhPLUhCHLlQhBLLQiVVBiYZljGoj7/Qdw5pgaMuAZ8SDOwq39DxPMhzTcsIXlyhXlj4d9FXv4abjcDO8NDUa1UYlv72b+jJwDTECGAIVYBBtpxSiOr8AarBG021cguGyob6BuI5A102dr23W1kCuTdEfzqDIwwrSrSnyr3zbH2W2vHCGnw2owoVeDep9vMdK7Vz35GTwS/4VK9wcB1T/uzyWyM4c+WFZs0rKR5XErzDO2LTWkAbrUIc2XdVNYTZcDdbGzdgdndN5ID4N9u7l9kZXsJxIANexvlTJQn/B1s2J9S+9uMby5ury1s/e22F68FvhFjIPSPflmkeGuwKSFS1VvzbTWUpbRrWNmbXJ5giIseLGu5/e8o/K+PyeXs7/dn7x/63zsMqMdFtjAqSfOclRB28aV13peguGFjp7Rmcl4urOdcYm9n3oz1CDeDKdGWFM7vFsw95o4wZVVxg0aBAm7GzwCmMfasGGbLx/z2bHp7/+cnniPFWDfC2CnYV6Qx4dL4WglYm3jkDu7eWez8QCRFf9u6m9kBeTW+1hm+4+A7xy72y1IGPJZqZAcI+hoOSromiEXBxvJ66XOrv72P2KfH2+Lu4RMGWfh9B5OjdLEEnhxXftpiuuB5YBeCyWgof7q2VLHZV+kB3BIiXVDfGIfrDz2s+JQxvfMnoZM1+BGRUENwLpZHue+dj1kVUP9c32SoLRlasbxhvTqcTFfB9UjB3VUS0LyMx2GEzg7ohJxD92atSohVk5dBiyGHFiShpXzqpRINTxLky3S9lKKhWgYHHwg4RNqAi2YpRaNAx7ANdKxo3UPDkRsZcHfGlea2zHBTC615H61W+eXl/W1raMYFKfzTwZ0peRD8G4bE6QLGPZG7AvhI+7Qmb/nHRJC/IOxFhJ6RaoBrQeSgu4wYRHAZVszUDjbyR2K35s9FcLCYvS2t5WDoV+XA9Uq76IocBn0uh3laxP4O9EJFAyDk7a64vn9j8b9JzGFECWZ+z4CoBwO8V8YFJuavBAb85hxQ+JCSsbC8YI6TnIrW/2BcIPAhXixgohjREg/ISvwuhdvYn7wq/hvy+I+508l0NmHhZDLt++5ktfJm/sqdrYcrNhu46+7473AYMKBMzMh+D2vvm9ATfwLWCEsxnxPjyQ4WnC4e7Fsj4xmkaRZiJh13AMTKaYaP/o2RhXb0uw3wb1ycK3AKmrGV3Arar2HlYy1VkV/dsSyG+4QHJhLHCqEmL5GYzBckLaWVMrOZvOPSPxQUuP3xYKyJTcbOx7pD+liSiAZcRDc+qM2GSSG9g4DYcLBXt/bXs2ZSne/S8CAJSUb2+A/rOzGS7KtbaxeXhrX8/+QVT9rdom3Yxw0BmO7dddgp4iLhcop4m6/pEhfTcIlXt8uzuoc85ArxR9EVjtpcISxl1eoOuTwvNtHGT0tlczsMKiiZb9WEzW/ggLXlv2Ua2jn3g1ktFS5lb5pYw2XqS0byki17EdGtkyOsPkYr2rfqcYiJOB4G38zh2qBQb3UHaiD1KtM74qZ3GLrDcOb2V6OpHwyYNxn4w0GwDtez6Yq5qyNMLwdDegWr6jDi6nDu1zVBff4zhTE3Pj6MoQVzDlcp9dJ7uJ5bMXwiU2mO3jdumTloetS0CgM/Dy8VlsowHZMZJWvE582UQK5Upg1Ui3vUGMugwKGJ+g1G5DCP8dwHxyzMhqGyQVDuIkwGmOyB2N8zqnTj8naZ6VF/ctp3+4OJ/QWvt7fWET4LLjc8s0+F47Ak2hQqp1OwnwVmhh7JdI+F6Y4S07eeI7B6SAkGdu16KnYYlZSa2O2rlgoqVXJ+UIlt1TAACmmZ2/82+gn7pGwaAfvhR+rQYxjC/MdKYgQQNXBwRLRKRV2ABTGnGKnd3Mr7YEHwdArm3qcg9inKsLzGtzGHv73B0B2bxSkTBnlcUP8Bi6wy4nCbd6x4Y4LeL+Tw66lxpD07XzjtWeXJjdJpZ9WD/9yzUloA2gXki1cR5OomJJZY+JGnnZ46f7khsFb+4zGu2qjPc0JcTuiomTRXdLBKmMsk5dvAPSe/YqiZNxz7gzEbzdhk6IfB2F1PPJ+tR95gvR6sp92GWiIwvmq+rhjWTPVEmOr00S/2GXuSqrK4/9/IZw9AVzSpEpMdjRo3qguXOhahRWXTPr/9ctHjKfRWDcnrJW66Gu2KEHRN5V3pZZGu5WyW83bupblRq4I/SBpS1UgktO4JMPYSShEbZa7aZu00rO7gFAxk3xMPI9UB13nK1/mvUi0w6ghurh2m269+BiymrM8byUXBN3khQX4UV3q+SYvUQbOoqVtVM3udEPn4axSHrbkmw6T+gjEFmAnXjWvO8ffbUP6kW01AZkjJhpc2YnmJX702mD/dbg+ik6EHGt9hMGdcRJ8iX7k2QU1VnE/BvBe84yrjlPu3q2SDWekQRQISItIuyLdGWEV9okKE/RGgJ4TdVz9p57yArKqytrcVfqrAOPjT4uaPabr4LZzURRHBZ7atxNRGJ0YDtRjBzuMpAvLAJ9mlrCeLW6fBevvfYDf/K2XOcOC4E2cIbm6iQGlWe+CqqRv2uXzPvkfJD1glFXZebXdRRtyvc7iRHMUvRUr8HVJ1Bzchs+ZlvnFk45KhRwK4UgphMb+pyPFwv0wl6d5yKyaJ8tcIlMf3YXxscCZRq6oy1Mm8ToLezWKOBL3R6XDouVMwi+qhjG03pHQNoJi0KPJgQ3ITgnkAM/aQpSWmKtItE81L9uL67NbibVBHcYNlVVF3iQH6UQk2jBl4hsDyt6sIwEax540je9tI7AvOZAvSaM/BHYM2AN+gCcutFVFRsj0abrwuL/8l3G1jqYMyN5x9VCHJ84aYDorkcAL48NYde9PJcIY9SYz/vmiIKTYZ8wtOekc5DLgcLjZMmT+Dh+ScETfzHax3BCHGvNgHG8lCvYkeNwAqF0FE4OdfFYJSRUTy6zvz68IEko+No8cE8bJ9crZimeIcQmyu3rF2KUgE8dnJL2TW2rLhhv1rqbUfIVEJumSqi0yf2b/RCwgYmOk1LVvDn6VlsSuL9laa4bSXj0C4EwfE5fTdmTt2fhBDEC7NiyrUUqNCt6oVHHIxLzYpmLICYlktbMJkT+A6QoD6Bc9pUteTQVCzFgjCiHVW5+5/rpUmO3kUHeXIa9iRxDYyKKa5YHWH5i1wHLgIQJUdtCQhfwUjK37ToJHSPPZy/FjTJNb9nHz3BcwK/hOyIFFrLTQ5sx2AZNwfwy4dTQfUlYol5DDDyNDkUQxHXFx/6l4oHmXnVdogl4rJG7xStJ72Tot6NG6mXirSMjnmVfPTKCB3y0q00e2FvPCBIZpHGwFfU2Z2ZzS8VYVUEZe5Z18oLwX3eUShUrC8/OHHlbxbG9GtLkPakrMRkc8xd+soJazKkdImHmA6tI9/+OCWbsH9QNCFe0YCimZbU0NsmDqVDZoIMcMSCUxoOxtZtMX8+qoO/jFEKwuZvpIe8kzxYHhuxDCpSrbyYmLlQMgBLDkYNZPcRAn2WPWP8m80EEs9PfbbWn47U+1GooVIbUZHnWrl0170x48fpzkm16JHMPcx/V62J6EHajkdI5x30rWDX+i5w+loxv+/P+01jXJ7+9TQ48rxl089cX+VfiJbeylbU2lqVJulEkyYxNYzZPvmOhp/YgrPiq6xiLVt9IuzxeK6d3F28XDdQ1MewzNC/Bfbt2ieeybwM/KjDaxoBOjyvUo5Ex8du4p5Bv+ZCESpCAgLvny8WbY2ZtOL6CwxczUVJQjaWv87trzXH572R+PhtClepUs1+U64fK/9Rz8rD3YE1ff7G+sdQkN0hbD1bgDXFHy3I3akDXqWA+TIqTh5Szy8F3jcVmS8AwwTUiDNUTAHZuoqnaC4jQ7e9LmtNpvXQA5K1HhMRdax4D42gCYLCLnp1RhUsh0idwsTfPDQxZ59kCB5gCR9YnGxl/jYl1wkQQ8aTrkwPyhBzu+vF7dvai11+DURgEuZipkW5HjN1PCFsTlqRvwQj+ga9+v82eoo9UsrAZuF0Up+2tyhfJuETTpka7GjDUk39m8dR++k6NvteNtwD7OkbkW5UQFRaVd4NcYtgqSJ1PItMEjYQG10zKiGcPYTfjYifUKBAOBWRqAXwg/i3A6CLopn5pdhVBzbRsuFMx7P3AGGwB4CPW/geVP75CHdWX6IwgSn8hZZnjhBhaz+vgSbUNrW1RMo9DmLY9t6X0bfwC4QDRB16q/SWpTJCSrijCviAuOkJHVu0r/9RKf6Q39HuI4rxCEuvH3x5fpzJUw708SKY5ktJwvGIFIrkxz+qzcN9bMZRvVE31ic/nDgxr9zURDS/ye4LcaYAVsn8iXdxTepdKaCCVZhztDVFaxd0n3X6307DZ6iUgi5P0JK9D3DfI5JT6m4g1FfYPofuleaJ8WfJUjDIuNfz9k+RcS3T/wtqPUCYHRBhU7rBlNpIg/1MD+rdY7yq0gfkKuv9Uwpy1/Vb3OHs3jHoZ8ul9Kci44SjrhW+AJOdVuY3gER27VCqcRhV3LyhYFXCzBWt9JaEUoWGXnr49tm/xq3jSa7x4zbla3Q3qpaOjjQzYrSQjUanw7Hs+loPDFckkhwajV6S53LaVgigqV7/g/LElhK07CB/u6iOEWvNRJZtvt0y7K0Jb9T7zOD735YXNeHMpQU4otxTCIlVI876Il5Tse5onaW0HR3bfDkwx7uWbY11YPMhXxdE616mAknuGrfKaIGXdfsTtXpPHnDHAWGL+T51yqKrXmElzi3RZEhxZ68myOKSxW0WpFzm3UyuEOFmE8V6fZAI9gxuNEn3SGse+pOR5Oh1je4KfMWnjhNYGfM9RIEDNSsAXdDf2IpiNLHzgc/2yoluZbzdiSAUeM8CAjJsNK5MKiGxpM2lWV+eaVM2E6AoV6he+LxfbJMLb3xEBSjU4dr0ILyIgTAYh2ymqoH4TLcR7X3Uqfr1iCyFZgHHRkH6tPsKL0xsfAxdusjRzyYwHA2uMbiZjBFikN6fDFvQtyHMXTCdHeqkiZnIPA+TKVRvLlJrdhJN3T2TvsTd2zojaF5HD2/1XshyCBWFgUfvA0QXbrOYcWpt4KHAJyyj0o1bBsowtMjBCAOJLaeGyFicIUW973b++t6GisHzeDt07s43ZMpCTORxUIUicAlRAub8yyK2qH8Uzf+isXOGvt4sWvOoR0q+o3UrdvzZN4zaxBqkFJXWHwoKyLGxDli3k5HtYE8l+j2E/NXqPAg+r7gyQJswmd5lREK0RP4NFRvGlNgOkPSenPQDD7wyyfNYP7MHelsgV4mXBU5SQ3VYsTV4p49prUSbaUa25jIhVkzBDA6KdpKLW1UXLlvw3rH4p2CPVqq+OolGi8x9UegA/ROfBepGQpUhpA8eaPlp4lpDxZgKZ/fhXNFk5KRxcdHfsy4FDOIM2jjNSANSbHiXRSMERH+0SAGF3U09YZ9Ltl+vz8aI66HH/MxXkfqgZy3yBfL0e2naCsCFvIweyQaCN/rYvx5nAbfrQvNsZ7H5aMDt+bMY38vsx+HQuaD1FIdHPZ2spEGrcNCDF7DkXEYS/PYJgLUIXql0aI1uYS5/lobrZAexiSM7+TD7OBh6qX6zlJjuzVQicmj0qQ6vNchcAUUi41/VCD8ikaA9dTtB4OV35+sfDaZhms2CwfjYByO+sF4PPSeZ6OihwkwvwUqn8xr0Zbgwb1P8xw337+sryzC5Jac4qEC0prNkFMWDykUlaKFc8FYQuZZRSk0b5nz13sHt4zVZiNHwxu9lHZEf5tKI7tqtH3k3UAa9emYupRElRqGPob5cVClSqLcgCFsV6hz4Ue46us+oNf6keNzLbLIPRp7g6ktoLUv1sOp5QJ56tkXijGRfSLaxByewVkrz/Bg6DMyN6KEyEGV6Zz0nUiUQUXdqzIBzX64vbnvgiY5KClE4b2QrdPHY4s0ucaiwtc0p9fJu80UauXAI8UeNKVAvcurT3KgY1sJ/aAGNd6UagQanIj5tR3DolD9t7AvKYeKDMEoESQqjY90wq7Vhmk9E5qX26qxQhfnDowsamsc/feHq823y5/zy1glzY35f+rOqpZHcP7uWQL6rLLFFafVkufh4FNqWTV4Fql//Lyorre1+xpq9FjjxMiACcctC4IWC414V0BzXMj6ZXm7ptIkNLTN+JlvFJomw8FlfgRfqQ55XuTLauNkkPqoTCf8jxApU7QqrvHmPCujS3FH8/MqnAycOpP7cbt/i4I4e47BO/AH08loOhlPPXft+d5o6K3G/XEYTibedBi6Bxi8aWZMs3NU6ZBzhFC/BGHx7rHcs8rUxS/RiuVvaK4JPgcW/nAEkPNFPjZFSQvYOXGMCdoK3Q0pbisIzrYWH4Uqs4nv76tja/XncvVDEjc/mPDBL2BHgLz4HCLnSVUR+bzNymdVHV5VLM7Et2XkgjCK2rhwIluN29jeQdfA1vp1VVamJLy19ZMyDwCzdKZucMuA9qCDfmQpdh/V+ucCWjPB89Es8d/j7DBqAjEIUt0IYu/JwFZZARCGX0vIajpK7Edb0JmxyD1fpknh523dOcrpqYHUtz6s/pWf763V3nqACESPsqo38HQhbGkzRduQiI10zvml/TmYUuoZaFqCOR5211nJ9W6l1/TsdPo0OaJ9h890DNxW8m84qlagbVuiHEUTsfxDzquNjt4fjyf2rf8NTSEvaGJlLNkLXqkQidNUFLde5jy7/Hh+/UbDZZ33Pc+i8BG5gV04mXsFgxVIlOj76/Zxl7V8zdwMxiSBsOpvJEEdwzJLh2WRzq81IzZV09Xo2VAYeiJuVLaV9t9q5N49DO0giu49lyEm9oXxWOJWK+O3MHXc6sl+R/1SJGe806E3mY76E5tXLcVCcZ2pZIlRZUQm+CYq9dzEXBiRDsXgYEeWnGjmiNlGQokbIwCXRClNgkEX0jX77ib94cTITcUsX+/D5b3Shm0KiCFNkNGGo21FF6nyOeJ+HJp0Qg8p7scYKASvgFByZuQH9C4Km02mpEzav3G9x3zL8R2DhwZAdfkrH78BwWIuSF94mATXrsowm05b1DFRWg28OdIWEWnGnc769oV/e+6g7XirTb1eOG6jUJmGXJnM9rV3/lH9a9RdjHll3KZwKfToWyqtPKFy6Zyrff3HxZeuwnd3uxm49Z+qH80sefLRUtRC9Mf86n2Pf8JUnx+MGCu2vj2Td/GCDrc6bdl4RxkZxCCd08NzzPwrZgR8drtNqz3T3Bm39L6Z+SLT/rzY0PT7k17uugNv6ICo4c9sMnH2ikWN8jeL6Ho5tckRncYt9FijzenqCRkuoo1QGQWhOaJhR4U5RrdGW0Oq/rqUv/q4fovyhDwSwktXCc0n8DmLPmebpwyI6hKOlOD9Oyd25b7NuaNyyLxUj94W+V38y52dV836k0opH1IVlVfGLAk9Db9OR2NkpFary3S8SD1mk6kD2jF0nclwMB453hLAK03DBThUMj5oBinDCXUEVDM84zEfCPifTao80s3VJVkQzjBCkVH3wgE1wFAJo8pLZDQeVgkObA2V0P2kcVU5eOhzgr9JVLuGJvDPgotg/vf8RJx2xPRN9gwuU10djifQPGcscCqdeoSdH2V8HE2kztsoE/EqCQKD7bw6I1Tpg0WjTLuavrrlPuj3vakzWQ6H6oAmxzRTRivR2OPyvtjojB4ijlwRrSFOgE27XlP8a4xv5qagMdO7rX2yo311UaTgMflpLK1DvilolYiOVqs6akX0WhoNqKbjqPTE4o3TVIAvi/vnOy5fvu/FJHDOalnXeOAdTXx2G/flSLCJyzweeNMpQk6KWaYTs00VHrMiZjEa7w/GYj9R/StXP3dIuMfsuxKAIWKzed1I6RuSt2qS757Pzb/61K4JAGLDdEspgvzAuPcTJGbaolWz2vNsraJE7TBOgmxAiRNxE7g2lc55sTuPMwZV3Tii1ChUo2n3m9Pihdl/wbz4Nr34Op4NBqQYACD77mBif85pErSe3c5nQmOqjosIVUScwfBXfYb7j8Z215KrtrrLI4ao9EmnoqkDEqxzSuYrIssi2MDztZ4A0GEtiI/NU0XUrhbkvevFXNuCE0xI4fJgy436lm6PE0XIGKBrLy4R3KKFlemvgIHHqNyxKD8cZS9a+HTP2Ir5hpiiBhn74uLhS4cfsF+f+8IFHk1BH2R1eTQZ8cOZhBOq+AUpSVQHMZ0PgEBNH0zzr13+AU1IrKv1OsLyHp1YBJ/Hv1VLP8244ksdK4rArKfiLn3QlFALekFfK1PXOgQEu44Ae1mIgKEmZbWwyoj34TTnhP2qg1e4f+D03Ul/7Hj86bmhIxhkStATecsPPp5kA3Dfvvh0dglykkewfSqp69xshzAPCUOFtP+4va6msPFVAdr+R3zfaHRQ8R59TJYHyI8bhj1CaXC22CL6KYPrysvl6psoJ1x/OrM6zg5r6Uk80h7jj5qbz1CQt2330xrvw8vmwIvnd+RwOB27PVxSTlp1B6fD06lrX/IeTlozhK957lANJmcsREGKvOJVEmYsAlle5KA+e3sxdC5urudvOhwyn33Bshxs2T27viT6zqEDNIzmng4/ffWzQLfEeej6Qq3HrWGZyXTQGN5zvqmG7Sf2gm2RQxvkMmZ71t52+OKjg7M7PAUUVNIv8whRGodsxd4hJhenZNO0IaQH7mS7efWgMJO79VIlUKSfr9501J/ylCG4abDL5yh0gjUYw2xRgJh9DRzTRlTCNW/ArfQDS9jjo8J1HyBcLlKA7kjz+istG6Pa0VrjYAX2k2jHAL1EHxwWzH2smKWWPtniHo80vOCHj7RmDdWUZTyU8U8x+ES+JApc1D9l18nRBoVKX4+OUBSHnZAW62EpOPqhR4m5nPiTNdJoxSK05f5ub+479ILfNo0mRS1lPzmzj3NdkZ+IHBEjeVP11bpsdXS678AY14KLU81xFRyNQK6QI1ZIt1h5YpzgV13VpAH7l1mEU6MOs9ZbO2a1OOZIQ+XVhrZY/iwIWMz4xsB6A19ILFWYR8KqYAhvqyJ3fhAAv0/1hEhskxelkgIJhqcAjhB5ozX92DLCn9WiGSf/PSf8ymRK3lcv4L2h6hkTTPzjMjf9wcTxlq7Ll4cm3VrdZkDk9G7+7/8pvmkNkGjb4eBbbm5BZK1uxZxXsq8T9hPZ1B8VsRf0A6ByevFQHVHRWnfiH7Tlj7TRyTs611WOGVusUR0LMs61NWwk8XZmA2FLBeAQTeJYlbg7mNDttTMjmvZAVCarbdWU1JDszmdLSzgVZgrqMVP53oivtC4GENRSbRJ0yC3qx5hDwavvW/Y9IAgBSCzRJpdCb2OyQX2qD6HGA0bknTymjiPHSsf8M7OCqdAUYU4sKTSxT1SM2iyInG9BDQsysWYalLbnkE6I1kdbGVN49daqbi/Tfa7Ta/SrSuCqn3hp2pn2dpZazfIVfPLx6dAbzYbTmXA05hhieQJ4o3TpeUKn8KmwJVHQAVm8dgw+dZsbEScJHXRJQl6aNrr4ctsNOxrE0Lh1zIbZrSgZcSbZ2158vr8SgX+3K3peMV7JC8XIBKtyWUQzp78bT9J7D3iVNBnCM9wn1dBSm5uKU6rrhGp+OcYHzQZDz3Pc5Uymj8xpBvz8GlSDCVeDv9RssE+fHy4u3giZxtWT2R7MRLhWjeoZY5mSfoM83Gxt0jqAduTT/NbWQE4fL9QwHqYmNJrJbHXvyr4cshvHodFXdL21gBGRtalDkW1zbKpxwMoxEUpj0M7k1B1NvL4ul5voQ9Wcvak4Bn5VWh9AQ5mecXuhOZ7vohhsHG/H5cF1x4mgbdUD/g1xgFQjJKVefc5uM8PNB9R4MaOM5/gA9GTpVktUA5JVjkdKyd4Zec9rume+40FOznPNRh0w45lExDuqYIheIpNRhAWuXa39iMqdKo2iK0fNkTxqPHoju3Q4dTwaeH1qROljSDru21d0hLXizyhGpVFb8GZCBXJdUTbnGWNWVY3wkxOvkInYMQuwDhFkq4nxK7YmaQBYRE4/Jxqp99uG2Z+hPBNe1CImZH1MoODd0t/rJ4G0TWd4hbjFI8NKcp6Mwa6yO0ofglNwkA0pe45e5vb7p8P+cOBNXKSIiXMd4ZfN2VxiBxnSnvR5MuoPP/lW6lNbyIlyyRo1I9DBO7+gPO8Nng0vj/lBzsh1BxeltcvsIMlEzDwOGbJWVrWctqgcVRizOYjK5tliXkxSwxdUpUjersPhwatnKb8mQ6VqcUidcnhdYEfPiD1oCI/owxCV4DEnLCVmr7HqBIwBYNBwDxM//hILpdVBTMeuikV2zWNYpGFyWngoE1ccO4P1gpceP1nRMm5SDJJlhb3EfQ8PeRdivmp1LHeb03nFOA81MKhl9q+Yzm3QJiv3yH3Nq8d5HFd3bM3GmHalvZ/xkE685OQplPPxx1qaAlqSgJdy8ZZVAS+FgNtPovqcRDQ2Ju46hHgyaB4axw0KvzWWYa5Un3/U1EveRCnOJTZELo53PkDx/hUlaygXz1XxE5GM2xADHHbqAV7PfDmsa+aj79SjcxPUljgxDlGVze1tqfZqzounoF/XOIkv9NwAQt1wNvXHQRgA2lmH02DirVerQeD22XDkzPELZmPJLlwbWqTX0dQhkXz9IzKU6Kws0ks84uFMnkl5cKIQ159Gc0BrwvUZtWmxPy3WqX7Mrn2ORTE+jbDeDPBifmXLIUxHpkM+NYqKUoUafS66jvq2oz7T1pd0VPbV81xnsDQSbepgURM1th+mNZHJ2DLVoRApLrHvv0S4pI7CQ8aBoFrNmgOn2prWWo3bceZDTyMS46dM88gn3TWNCFYbraDCBv6dRqSRbRUjOOyODNjBoPel3YrF8ClJbpj/7bOstjxxQcVSUF0ORJzucqaqbye0yjTUXTgO7gtQWG+sK4BEPyAYlu6McxlFt0/0t2/M+3gXIVMfBxNXD91raMfJ78IwdzhhWba7khoc1hHjSbll5KYfn/Qf0A25SLkvXUuhiiNvX4d0u44Pa+fbTYZDnoeHP/0hJ9UG/s4PUEhlK9SoYl7RWI/VDqbplrzAgb6CflGawKaPeAc2JP2BVuRDucXsvDl4p81VGB9oIUXLyEl3tVYOsjgMNXKGQqKGIfMqv/kUxsM0qrmIisK0ds4Fko+pttIGevWxM5UBZLjJhHFXRw4dPBjDG3lTI1Bscs5NKzGRx73p8pwAmGdw0YLDhCTdYjTXFDx9hsqyxo7j54nUHEkX1ryozioytWBBjRW0FpUdnqZy8AA/cGWNRALiEBYtYad9e3Nzy+vnO5w8K7O84hwaNXZJKcLBfvgaiGgcE3DASvC5txncehoFxtng5mngYCtqkzINi9FBvyR9qlBddHvha08sIkwaTKZrbzgbDUaraTidBWw1Wg8YYNJRuGajdQcmlcgE1reMfT2MyjFluONrgQo4rbupS+SYXOAYL2F3LqQlM2h+HTh1nkVPfrB3Fv6aKdLYRXn48NfnTtGhBW4epZPprzVP0wmMurHGq5wn8IqDdQ6pFY9pStol/IAdnowpxEzBMqkfWNbCEjAKHb9hGoxQIG80HowmI6/vjgazcX+2HvaH4zCYusFgOBwMvHYFMtKvawE0OK/ENFuiX/6h1KySLe7s7jPpakj2hQcE6y82z677KA6pQGJAUDkVBnZlhqdxMe20fPM8dFNvzmtnUpM+qzZNkSt+4fj19lMwjtAqUQ1R9y9LFqBZNJ+BtxMAGuTD3iqd8mSa39ZH/FYt3a8cpDYYT71ZuFpPvOlgMBkMx5NVMAhWq/XU6zM3OHCQWr13mnfZK3zkcNFS23QOSjYVB1C/Y8o2Pa9eBFea1gnFbvQlteJi+W1Tt4hlI49fkmOF6sekKuXKalc0moxk8FQ9Tp5rmeq+r5638spZ/y+2YMbx4yakW4MGCZZrDujSYCc3Sw/SmKlZbSo587qkIIh8qWVtHLZLshYZQbrllnwhnk9vyLo9AcgLWKUQOe/CNprWtambuvLwRyy071QaB3EVOViToqTFW2W+tE3SO3gWel2buPY8tBevOXeOo2qWKROnqTarMoMv2LcKErQ4R02eMt5uVrYOURmOSAZWJ8xs9WQePTES65w0dXPfDbtfYbZWAzyo15/NZqspmCtvHYT9od8Px/5khGf1dpst3TnQmvmqKIs8jRu5VIAMWJT53ZOwELdrjSE3hqQHkc18Zv5H/Wiu2vlNckCi0pGb6mAiOssHT8USg7BoXLW6l1Tci8ys1oae/WatOBPDVv36MK4yrzTMEW4STqLOcGk/Xt04x+gXVGfqeePxKhi7njuasqHrroJ1sJ5O1+5kNArZpFt1asJy+BiwisaIHPG/S2TfbSJbnEaxl0ce8LOZOFPz2aHm1YlYL7E6XWolX2/OkuE9+kcoU2UcY9ug6KZ2dYR5r1Av3nhNS4kZNTXYFzMBxqyuiorJVLiZJBAzHLrHX/0e3M7PFF/7g3Cw8oYTdzDoQ4g2GvsDiPsmwcAdjYerbn3T80HN4+ky/fgVevB0pM411vWJzwmEfvmGmPnYqi15VBaGgSIJSQeZNeFXl6F7qc4dHGFEarevzgGY83HmzMTvzaPr/jH7xQlZmmy71ZU/ZBIqE9XR2MmP0KEQUXAMD45KO1KJxqO1Px2Mx0EIYd9k6LHAD4dTNh0BZh+zyYGD6TFud4zxPBVzJRnDMY4I0I3h72jR3/D5RdTIqamePh+pnzr5JjVPqG8bOf1KVTFxq0ZFariw+bYe9dl6pFLnRHvJ+aqho5b5e6+2UsZ4CP8JHNizzk8YpVZu528bLTsWRmk49Car2XrtzmbhwBtM++HKC9bj4Sx03cE46NYnUTdJ25ZbL7KpZJJCnK2w5+iogaLvbs6u738RM1XSSJrwzSM1JHkCPi7sd9jlxvep0iX128rYGG/9fodm8mKLEgw7WIxHhPayBGL2MEqa8hNmR+DnABO8VS5Ow+7qRH5K+BhHHvyK5gQTn3ls5o1n634wm40Bfg/YFFzZyu+vRu7wWc0JKwapMEutVX8mOMd/+HuMqFW7C9kIQFQXclBUBUqpPOPhRLmwNB0GqOadhM+qOCnz6sZQKzUJiXKT/6yjauXAW7wjWQ2x1Z3xIm1Un/xbxTz6XGOsBVdODX+FroSzQTj0WeD5gLRnw/Vs6M5G4/6QsdFs7bp+t6688/PKdKBODyYS3iax5+rnLk6jonrmqsY1jbx3g5/c6FLn2cQGK/mzObTENDfVERYG5ZD2pdFTG6d5fkKHgMYM51VU80lH5bZflor8nDMy2YUYhy1zwlI7WqrtyPAzT/voqMHWQjbtp57vdB/Bn/5s4OzbstWYCQmZk67XNfMgktZfS92UAmv4xpJzrnTTMT+O53m5t6URjXLG8fLnJY1a/7T5AXlvjqina8EfM4jixVI/kdcTFGhR06ChVbpH2ihnKBbOwR5oAWwbZfWjcoMjnhv8Wi5JbkshtmVNbEstNpEcrItNZQRHZkZQHb1SPe6yXvqY9WXQpCdhfd7leKS9c+YQbRk0SswEQCslvVat1bpuWV4wN8nQsI6GfKlnzcF6J/dVW/IM31k+SLy3WlHycTNQXqh7r2enyuMLnd9HUxVah+Je1uS8bBGzkasmMVOu2tBI/sUjNVKUco1KpwF9SLaOOc/p/wE3z3CJULYAAA=='''
EXPECTED_LITERATURE_SNAPSHOT_SHA256 = '3596bc71e6ff585b749edafb0845dbd88a324e3c9c811b1ca395ca9ffe073387'
EXPECTED_LITERATURE_STUDIES = 90

literature_csv_bytes = gzip.decompress(base64.b64decode(LITERATURE_SNAPSHOT_GZIP_B64))
assert sha256_bytes(literature_csv_bytes) == EXPECTED_LITERATURE_SNAPSHOT_SHA256
literature = pd.read_csv(io.BytesIO(literature_csv_bytes))
assert len(literature) == EXPECTED_LITERATURE_STUDIES == 90
assert (literature['Closest competitor'] == 'Yes').sum() == 62
assert (literature['Peer reviewed'] == 'Yes').sum() == 81
assert (literature['Inclusion decision'] == 'Include').all()

with zipfile.ZipFile(ANALYSIS_ARCHIVE_05E) as archive:
    prefix = selected_archive['prefix']
    def read_json(name):
        return json.loads(archive.read(prefix + name))
    def read_csv(name):
        return pd.read_csv(io.BytesIO(archive.read(prefix + name)))

    STATUS_05E = read_json('status.json')
    CHECKS_05E = read_json('checks.json')
    PRIMARY_05E = read_json('primary_hypotheses.json')
    RECEIPT_05E = read_json('unsealing_receipt.json')
    QUALITY_05E = read_csv('quality_summary.csv')
    RISK_05E = read_csv('risk_summary.csv')
    CALIBRATION_METRICS_05E = read_csv('calibration_metrics.csv')
    CALIBRATION_BINS_05E = read_csv('calibration_bins.csv')
    SOURCE_MANIFEST_05E = read_csv('source_manifest.csv')

assert SOURCE_MANIFEST_05E['source_id'].nunique() == 40
assert not SOURCE_MANIFEST_05E['source_id'].duplicated().any()
assert set(QUALITY_05E['chain_id']) == {
    'q8_b16_n2', 'j90_b16_n2', 'j75_b12_n2', 'j75_b16_n2',
    'j75_b20_n2', 'j75_b16_n5', 'j50_b16_n2'
}

literature_summary = pd.DataFrame([
    ('Studies in locked snapshot', len(literature)),
    ('Peer-reviewed studies', int((literature['Peer reviewed'] == 'Yes').sum())),
    ('Coded closest competitors', int((literature['Closest competitor'] == 'Yes').sum())),
    ('Studies with calibration = Yes', int((literature['Calibration'] == 'Yes').sum())),
    ('Studies with abstention = Yes', int((literature['Abstention'] == 'Yes').sum())),
    ('Studies with operator UQ = Yes', int((literature['Operator UQ'] == 'Yes').sum())),
], columns=['Literature audit item', 'Count'])
show_table(literature_summary)



## 4. Apply the fixed synthesis rules

A confirmatory claim is retained only if its predeclared statistical and practical gates
passed. Calibration cannot be claimed when the locked failure event has no positives.
Secondary chain and risk-coverage patterns are descriptive and cannot rescue a failed
confirmatory gate.



In [ ]:
h1 = PRIMARY_05E['hypotheses']['H1_reconstruction']
h2 = PRIMARY_05E['hypotheses']['H2_selection']

quality_pivot = QUALITY_05E.pivot(index='chain_id', columns='method', values='mean_detail_mse')
chain_effects = quality_pivot[['dpir_nominal', 'fbcnn_dpir_nominal']].reset_index().rename(columns={
    'dpir_nominal': 'dpir_nominal_mean_detail_mse',
    'fbcnn_dpir_nominal': 'fbcnn_dpir_nominal_mean_detail_mse',
})
chain_effects.columns.name = None
chain_effects['relative_detail_mse_reduction'] = (
    chain_effects['dpir_nominal_mean_detail_mse']
    - chain_effects['fbcnn_dpir_nominal_mean_detail_mse']
) / chain_effects['dpir_nominal_mean_detail_mse']
chain_order = ['q8_b16_n2', 'j90_b16_n2', 'j75_b12_n2', 'j75_b16_n2', 'j75_b20_n2', 'j75_b16_n5', 'j50_b16_n2']
chain_effects['chain_id'] = pd.Categorical(chain_effects['chain_id'], categories=chain_order, ordered=True)
chain_effects = chain_effects.sort_values('chain_id').reset_index(drop=True)
chain_effects['chain_role'] = np.where(
    chain_effects['chain_id'].astype(str).eq('q8_b16_n2'),
    'uncompressed control',
    'JPEG acquisition chain',
)

observed_bin_rates = CALIBRATION_BINS_05E['observed_bad_detail_rate'].dropna().to_numpy()
zero_positive_events = bool(len(observed_bin_rates) and np.allclose(observed_bin_rates, 0.0))
assert zero_positive_events, 'Locked calibration boundary changed unexpectedly.'

primary_risk = RISK_05E[
    (RISK_05E['chain_id'] == PRIMARY_CHAIN)
    & (RISK_05E['pipeline'] == 'fbcnn_dpir_nominal')
    & (RISK_05E['region'] == 'all')
    & (RISK_05E['coverage'].isin([0.50, 0.75, 0.90, 1.00]))
    & (RISK_05E['score'].isin([
        'oracle_detail_error',
        'operator_spread_detail',
        'image_transform_spread_detail',
        'trained_image_only_patcherrornet_ensemble',
        'expected_random',
    ]))
].copy()

claims = pd.DataFrame([
    {
        'claim_id': 'C1',
        'claim': 'JPEG-aware deblocking before mismatch-aware DPIR improves detail fidelity under compressed acquisition-chain mismatch.',
        'decision': 'SUPPORTED_WITHIN_LOCKED_SCOPE',
        'basis': f"H1 passed: {100*h1['relative_reduction']:.2f}% reduction; Holm p={h1['holm_adjusted_p_value']:.6g}.",
        'permitted_wording': 'Independently confirmed for the seven-chain locked experiment; describe the compressed-chain pattern and uncompressed control.',
        'prohibited_wording': 'Do not call deblocking, DPIR, or their serial composition generally novel or universally superior.',
    },
    {
        'claim_id': 'C2',
        'claim': 'Operator-spread selection achieves the predeclared practical selective-risk advantage.',
        'decision': 'NOT_ESTABLISHED',
        'basis': f"Statistically detectable but {100*h2['relative_reduction']:.2f}% < {100*h2['practical_gate_threshold']:.0f}% practical threshold.",
        'permitted_wording': 'Report a statistically detectable, subthreshold improvement and descriptive comparison with the learned comparator.',
        'prohibited_wording': 'Do not claim the confirmatory selection gate passed; do not change the 5% threshold.',
    },
    {
        'claim_id': 'C3',
        'claim': 'The scores are calibrated for the locked bad-detail event.',
        'decision': 'NOT_ESTIMABLE_FOR_POSITIVE_EVENTS',
        'basis': 'All reliability bins had zero observed events for detail RMSE > 0.05 across 286,720 patch rows.',
        'permitted_wording': 'State that the maps overpredicted the locked event and positive-event calibration could not be validated.',
        'prohibited_wording': 'Do not claim calibrated failure probabilities or use point-estimate Brier rankings as formal superiority tests.',
    },
    {
        'claim_id': 'C4',
        'claim': 'The unified evidence-calibrated selective-reconstruction novelty claim is established.',
        'decision': 'NOT_ESTABLISHED',
        'basis': 'The combined experimental gate failed and the calibration target produced no positive events.',
        'permitted_wording': 'Present the study as a rigorous acquisition-chain and reliability assessment with a transparent pivot.',
        'prohibited_wording': 'Do not use first, uniquely reliable, fully calibrated, hallucination-free, or forensic recovery.',
    },
    {
        'claim_id': 'C5',
        'claim': 'A narrower acquisition-chain/reliability-assessment contribution is supportable.',
        'decision': 'SUPPORTED_AS_PIVOT',
        'basis': 'Strong H1, coherent compressed-versus-control pattern, complete held-out audit, and informative negative reliability boundaries.',
        'permitted_wording': 'Emphasise what improved, where it improved, and which reliability claims did not survive.',
        'prohibited_wording': 'Do not relabel the pivot as proof of the original unified architecture claim.',
    },
])

assert h1['confirmatory_gate_passed'] is True
assert h2['statistical_gate_passed'] is True
assert h2['practical_gate_passed'] is False
assert h2['confirmatory_gate_passed'] is False
assert PRIMARY_05E['experimental_novelty_gate_passed'] is False
assert (claims.loc[claims.claim_id == 'C4', 'decision'].iloc[0] == 'NOT_ESTABLISHED')

decision_summary = pd.DataFrame([
    ('H1 reconstruction gate', 'PASS', f"{100*h1['relative_reduction']:.2f}% detail-MSE reduction"),
    ('H2 statistical gate', 'PASS', f"Holm p={h2['holm_adjusted_p_value']:.6g}"),
    ('H2 practical gate', 'FAIL', f"{100*h2['relative_reduction']:.2f}% versus 5% required"),
    ('Positive-event calibration', 'NOT ESTIMABLE', 'Zero locked bad-detail events'),
    ('Unified novelty claim', 'NOT ESTABLISHED', 'Combined gate did not pass'),
    ('Narrow acquisition-chain contribution', 'SUPPORTED', 'Proceed as transparent pivot'),
], columns=['Decision item', 'Outcome', 'Evidence'])
show_table(decision_summary)



## 5. Inspect the descriptive robustness pattern

The figure below is secondary evidence. It supports interpretation of H1 but does not
replace or modify either confirmatory test.



In [ ]:
OUTPUT_DIR_05F = RESULTS_DIR / 'independent_05f_locked_synthesis'
FIGURE_DIR_05F = OUTPUT_DIR_05F / 'figures'
FIGURE_DIR_05F.mkdir(parents=True, exist_ok=True)

plot_frame = chain_effects.copy()
values = 100 * plot_frame['relative_detail_mse_reduction'].to_numpy()
colors = ['#8c8c8c' if role == 'uncompressed control' else '#1769aa' for role in plot_frame['chain_role']]
fig, ax = plt.subplots(figsize=(10.5, 5.6))
bars = ax.bar(plot_frame['chain_id'].astype(str), values, color=colors, edgecolor='black', linewidth=0.5)
ax.axhline(0, color='black', linewidth=1)
ax.set_ylim(min(-10, float(values.min()) - 5), max(105, float(values.max()) + 5))
ax.set_ylabel('Relative detail-MSE reduction (%)')
ax.set_xlabel('Locked acquisition chain')
ax.set_title('FBCNN + DPIR versus DPIR across the independent acquisition chains')
ax.grid(axis='y', alpha=0.25)
for bar, value in zip(bars, values):
    offset = 1.8 if value >= 0 else -1.5
    va = 'bottom' if value >= 0 else 'top'
    ax.text(bar.get_x() + bar.get_width()/2, value + offset, f'{value:.1f}%', ha='center', va=va, fontsize=9)
from matplotlib.patches import Patch
ax.legend(
    handles=[
        Patch(facecolor='#8c8c8c', edgecolor='black', label='Uncompressed control'),
        Patch(facecolor='#1769aa', edgecolor='black', label='JPEG acquisition chains'),
    ],
    loc='upper left',
    frameon=False,
)
fig.text(0.99, 0.01, 'Descriptive secondary analysis', ha='right', va='bottom', fontsize=8.5, color='#555555')
fig.tight_layout()
chain_figure_path = FIGURE_DIR_05F / 'independent_chain_reconstruction_effect.png'
fig.savefig(chain_figure_path, dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig)

display_effects = chain_effects[['chain_id', 'chain_role', 'relative_detail_mse_reduction']].copy()
display_effects['relative_detail_mse_reduction_percent'] = 100 * display_effects.pop('relative_detail_mse_reduction')
show_table(display_effects, rows=10)



## 6. Write the final synthesis package

The generated package contains the claim table, exact input receipt, literature snapshot,
manuscript-ready wording, checks, figure, status, and cryptographic export manifest.



In [ ]:
timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')

nearest_competitors = literature[literature['Closest competitor'] == 'Yes'].copy()
strongest_calibration_rows = CALIBRATION_METRICS_05E[
    (CALIBRATION_METRICS_05E['scope'] == 'all_chains')
    & (CALIBRATION_METRICS_05E['metric'] == 'source_macro_brier')
].sort_values('estimate').reset_index(drop=True)

risk_50 = primary_risk[np.isclose(primary_risk['coverage'], 0.50)].set_index('score')['mean_detail_mse']
required_scores = {
    'oracle_detail_error', 'operator_spread_detail', 'image_transform_spread_detail',
    'trained_image_only_patcherrornet_ensemble', 'expected_random'
}
assert required_scores.issubset(set(risk_50.index))

literature_boundary = {
    'cutoff_date': LITERATURE_CUTOFF,
    'studies': int(len(literature)),
    'peer_reviewed': int((literature['Peer reviewed'] == 'Yes').sum()),
    'closest_competitors': int((literature['Closest competitor'] == 'Yes').sum()),
    'source_gap_map_sha256': EXPECTED_GAP_MAP_SHA256,
    'source_review_matrix_sha256': EXPECTED_REVIEW_MATRIX_SHA256,
    'embedded_canonical_snapshot_sha256': EXPECTED_LITERATURE_SNAPSHOT_SHA256,
    'boundary_statement': (
        'The broad ideas of physics-informed mismatch handling, blind image/operator inference, '
        'all-in-one restoration, diffusion priors, and uncertainty alone are already occupied. '
        'The original unified evidence-calibrated selective-reconstruction claim is not established '
        'by Experiment 05 because its combined gate did not pass.'
    ),
}

synthesis_decision = {
    'stage': '05F_locked_literature_and_experimental_synthesis',
    'status': 'completed_locked_synthesis',
    'experiment_id': 'independent_05',
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
    'independent_evaluation_complete': True,
    'literature_synthesis_complete': True,
    'experimental_novelty_gate_passed': False,
    'final_literature_novelty_claim_established': False,
    'original_unified_claim_retained': False,
    'narrow_acquisition_chain_contribution_supported': True,
    'recommended_pivot': 'acquisition_chain_and_reliability_assessment',
    'h1_reconstruction': h1,
    'h2_selection': h2,
    'calibration_boundary': {
        'locked_event': 'centre 16x16 patch detail RMSE > 0.05',
        'patch_rows': int(CHECKS_05E['calibration_patch_rows']),
        'observed_positive_events_in_reliability_bins': 0,
        'positive_event_calibration_estimable': False,
        'interpretation': 'The maps overpredicted this event; positive-event calibration was not validated.',
    },
    'literature_boundary': literature_boundary,
    'next_stage': 'manuscript_and_repository_checkpoint',
}

input_receipt = {
    'stage_05e_archive_path': str(ANALYSIS_ARCHIVE_05E),
    'stage_05e_outer_sha256': selected_archive['archive_sha256'],
    'stage_05e_expected_original_outer_sha256': EXPECTED_ORIGINAL_05E_SHA256,
    'stage_05e_is_original_notebook_zip': selected_archive['archive_sha256'] == EXPECTED_ORIGINAL_05E_SHA256,
    'stage_05e_internal_manifest_sha256': selected_archive['internal_manifest_sha256'],
    'stage_05e_manifest_files_checked': int(selected_archive['manifest_files_checked']),
    'stage_05e_sources': int(CHECKS_05E['sources']),
    'stage_05e_sealed_shards': int(CHECKS_05E['sealed_shards']),
    'literature_boundary': literature_boundary,
}

manuscript_text = f'''# Locked literature and experimental synthesis

## Scope

This synthesis uses the one-time independent analysis of 40 sealed sources across seven acquisition chains and the literature snapshot locked to 15 September 2026. No post-test threshold, endpoint, method, or hypothesis was changed.

## Confirmatory reconstruction result

On the primary JPEG chain (`{PRIMARY_CHAIN}`), FBCNN preprocessing followed by nominal DPIR reduced source-level detail MSE by **{100*h1['relative_reduction']:.2f}%** relative to nominal DPIR alone. The mean paired difference was {h1['mean_difference']:.9g}, with a 95% source-bootstrap interval of [{h1['paired_source_bootstrap_95_ci'][0]:.9g}, {h1['paired_source_bootstrap_95_ci'][1]:.9g}] and Holm-adjusted one-sided p = {h1['holm_adjusted_p_value']:.6g}. The predeclared reconstruction gate passed.

The secondary chain analysis was directionally coherent: the serial FBCNN+DPIR pipeline improved detail MSE on every JPEG chain, while the uncompressed control changed by {100*chain_effects.loc[chain_effects['chain_id'].astype(str).eq('q8_b16_n2'), 'relative_detail_mse_reduction'].iloc[0]:.2f}%. This pattern supports an acquisition-chain interpretation rather than a universal advantage.

## Confirmatory selection result

At 50% coverage on the primary chain, operator-spread selection reduced source retained-patch detail risk by **{100*h2['relative_reduction']:.2f}%** relative to image-transform spread. The paired 95% bootstrap interval for the absolute difference was [{h2['paired_source_bootstrap_95_ci'][0]:.9g}, {h2['paired_source_bootstrap_95_ci'][1]:.9g}], and the Holm-adjusted one-sided p-value was {h2['holm_adjusted_p_value']:.6g}. The statistical gate passed, but the predeclared 5% practical gate did not; therefore the confirmatory selection claim was not established.

Descriptively, the 50%-coverage detail risks were {risk_50['oracle_detail_error']:.9g} for the oracle, {risk_50['operator_spread_detail']:.9g} for operator spread, {risk_50['image_transform_spread_detail']:.9g} for image-transform spread, {risk_50['trained_image_only_patcherrornet_ensemble']:.9g} for the trained PatchErrorNet ensemble, and {risk_50['expected_random']:.9g} for random retention. These comparisons are secondary and do not override the failed practical gate.

## Calibration boundary

Across {CHECKS_05E['calibration_patch_rows']:,} calibration patch rows, no reliability-bin event exceeded the locked threshold of centre-patch detail RMSE > 0.05. Consequently, positive-event calibration and discrimination could not be validated. The non-zero predicted probabilities indicate overprediction of this locked failure event. Brier-score rankings are descriptive only because no formal pairwise calibration comparison was predeclared.

## Literature boundary and final claim decision

The locked evidence snapshot contains {len(literature)} studies, including {(literature['Peer reviewed'] == 'Yes').sum()} peer-reviewed works and {(literature['Closest competitor'] == 'Yes').sum()} coded closest competitors. It establishes that physics-informed mismatch handling, blind image/operator inference, all-in-one restoration, diffusion priors, and uncertainty estimation are not individually novel.

Because the combined experimental novelty gate did not pass, the original unified evidence-calibrated selective-reconstruction novelty claim is **not established**. The supportable contribution is narrower: an independently audited demonstration that JPEG-aware deblocking can substantially improve detail fidelity before mismatch-aware DPIR on compressed acquisition chains, together with a transparent reliability assessment showing that operator-spread selection was statistically detectable but practically subthreshold and that the locked calibration event was too rare to validate.

## Permitted claim

> In the locked independent experiment, JPEG-aware deblocking before mismatch-aware DPIR produced large detail-fidelity gains across compressed acquisition chains, while offering no benefit on the uncompressed control. Operator-spread uncertainty yielded a statistically detectable but subthreshold selective-risk improvement, and positive-event calibration could not be established at the predeclared failure threshold.

## Prohibited claims

- first physics-informed reconstruction method robust to mismatch;
- first blind or joint image/operator reconstruction system;
- fully calibrated uncertainty or guaranteed abstention;
- hallucination-free or forensic recovery;
- practical superiority of operator-spread selection under the predeclared 5% gate.

## Recommended paper direction

Proceed as an acquisition-chain and reliability-assessment paper or a rigorous benchmark/protocol contribution. Preserve the negative H2 practical-gate result and calibration boundary as central findings. A future confirmatory study may define a better-powered failure event and broader real-device transfer protocol, but it must be preregistered and reported as a new experiment.
'''

status_05f = {
    'stage': '05F_locked_literature_and_experimental_synthesis',
    'status': 'completed_locked_synthesis',
    'experiment_id': 'independent_05',
    'final_literature_novelty_claim_established': False,
    'original_unified_claim_retained': False,
    'narrow_acquisition_chain_contribution_supported': True,
    'recommended_pivot': 'acquisition_chain_and_reliability_assessment',
    'next_stage': 'manuscript_and_repository_checkpoint',
    'updated_at_utc': datetime.now(timezone.utc).isoformat(),
}

checks_05f = {
    'stage_05e_archive_crc_passed': True,
    'stage_05e_manifest_files_checked': int(selected_archive['manifest_files_checked']),
    'stage_05e_sources': int(CHECKS_05E['sources']),
    'stage_05e_chains': int(CHECKS_05E['chains']),
    'literature_rows': int(len(literature)),
    'literature_closest_competitors': int((literature['Closest competitor'] == 'Yes').sum()),
    'claims': int(len(claims)),
    'h1_gate_passed': bool(h1['confirmatory_gate_passed']),
    'h2_statistical_gate_passed': bool(h2['statistical_gate_passed']),
    'h2_practical_gate_passed': bool(h2['practical_gate_passed']),
    'zero_positive_calibration_events': bool(zero_positive_events),
    'final_unified_claim_established': False,
}

(OUTPUT_DIR_05F / 'literature_snapshot.csv').write_bytes(literature_csv_bytes)
nearest_competitors.to_csv(OUTPUT_DIR_05F / 'nearest_competitors.csv', index=False)
chain_effects.to_csv(OUTPUT_DIR_05F / 'secondary_chain_effects.csv', index=False)
primary_risk.to_csv(OUTPUT_DIR_05F / 'primary_risk_coverage.csv', index=False)
strongest_calibration_rows.to_csv(OUTPUT_DIR_05F / 'calibration_brier_point_estimates.csv', index=False)
claims.to_csv(OUTPUT_DIR_05F / 'claim_decisions.csv', index=False)
(OUTPUT_DIR_05F / 'manuscript_results.md').write_text(manuscript_text, encoding='utf-8')
(OUTPUT_DIR_05F / 'synthesis_decision.json').write_text(json.dumps(synthesis_decision, indent=2), encoding='utf-8')
(OUTPUT_DIR_05F / 'input_receipt.json').write_text(json.dumps(input_receipt, indent=2), encoding='utf-8')
(OUTPUT_DIR_05F / 'checks.json').write_text(json.dumps(checks_05f, indent=2), encoding='utf-8')
(OUTPUT_DIR_05F / 'status.json').write_text(json.dumps(status_05f, indent=2), encoding='utf-8')

manifest_files = []
for path in sorted(OUTPUT_DIR_05F.rglob('*')):
    if path.is_file() and path.name != 'export_manifest.json':
        manifest_files.append({
            'path': path.relative_to(OUTPUT_DIR_05F).as_posix(),
            'byte_count': path.stat().st_size,
            'sha256': sha256_file(path),
        })
export_manifest = {
    'stage': '05F_locked_literature_and_experimental_synthesis',
    'status': 'completed_locked_synthesis',
    'experiment_id': 'independent_05',
    'files': manifest_files,
}
(OUTPUT_DIR_05F / 'export_manifest.json').write_text(json.dumps(export_manifest, indent=2), encoding='utf-8')

archive_path = OUTPUT_DIR_05F.parent / f'{OUTPUT_DIR_05F.name}_{timestamp}.zip'
temporary_archive = archive_path.with_suffix('.zip.partial')
with zipfile.ZipFile(temporary_archive, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in sorted(OUTPUT_DIR_05F.rglob('*')):
        if path.is_file():
            archive.write(path, arcname=f'{OUTPUT_DIR_05F.name}/{path.relative_to(OUTPUT_DIR_05F).as_posix()}')
temporary_archive.replace(archive_path)

with zipfile.ZipFile(archive_path) as archive:
    assert archive.testzip() is None
for row in export_manifest['files']:
    path = OUTPUT_DIR_05F / row['path']
    assert path.stat().st_size == row['byte_count']
    assert sha256_file(path) == row['sha256']

print(json.dumps({
    'stage': status_05f['stage'],
    'status': status_05f['status'],
    'final_literature_novelty_claim_established': False,
    'narrow_acquisition_chain_contribution_supported': True,
    'recommended_pivot': status_05f['recommended_pivot'],
    'output_directory': str(OUTPUT_DIR_05F),
    'archive_path': str(archive_path),
    'archive_sha256': sha256_file(archive_path),
    'manifested_files': len(export_manifest['files']),
}, indent=2))



## 7. Final decision and handoff

The table is the locked reporting boundary. A `NOT_ESTABLISHED` result is not an error
and must not be converted into a pass by changing the practical threshold, the event
definition, the comparison, or the reporting population.



In [ ]:
show_table(claims[['claim_id', 'decision', 'claim', 'basis']], rows=10)
print()
print('FINAL PERMITTED CLAIM:')
print()
permitted_section = manuscript_text.split('## Permitted claim', 1)[1].split('## Prohibited claims', 1)[0].strip()
print(permitted_section)
print()
print('UPLOAD/ARCHIVE THIS ZIP:', archive_path)
print('ZIP SHA-256:', sha256_file(archive_path))
print('ALSO DOWNLOAD THIS EXECUTED NOTEBOOK: File > Download > Download .ipynb')

